# Transfer Learning Between Subjects

## Overview
This notebook compares three scenarios: zero-shot transfer, fine-tuning, and training from scratch.

## Key parameters
| Parameter | Value | Meaning |
|-----------|-------|---------|
| Source | Subject 1 | Source subject |
| Target | Subject 2 | Target subject |
| Fine-tune epochs | 10 | Fine-tuning epochs |
| Fine-tune lr | 0.0001 | Fine-tuning learning rate |

## 1. Install dependencies

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn torch

## 2. Load data for two subjects

In [ ]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

def load_data(subject):
    dataset = BNCI2014_001()
    paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
    X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[subject])
    mask = (labels == 'left_hand') | (labels == 'right_hand')
    X = X[mask]
    labels = labels[mask]
    y = np.array([0 if lab == 'left_hand' else 1 for lab in labels])
    return X, y

X1, y1 = load_data(1)
X2, y2 = load_data(2)
print(f"Subject 1: {X1.shape}")
print(f"Subject 2: {X2.shape}")

## 3. Build EEGNet

In [ ]:
import torch
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, n_channels=22, n_samples=1001, n_classes=2, F1=8, D=2, F2=16, dropout=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, F1, (1, 64), padding='same')
        self.batchnorm1 = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1)
        self.batchnorm2 = nn.BatchNorm2d(F1 * D)
        self.activation = nn.ELU()
        self.pool1 = nn.AvgPool2d((1, 4))
        self.dropout1 = nn.Dropout(dropout)
        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding='same', groups=F1 * D),
            nn.Conv2d(F1 * D, F2, (1, 1)),
        )
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.dropout2 = nn.Dropout(dropout)
        dummy = torch.zeros(1, 1, n_channels, n_samples)
        out = self._features(dummy)
        self.classify = nn.Linear(out.view(-1).shape[0], n_classes)

    def _features(self, x):
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.depthwise(x)
        x = self.batchnorm2(x)
        x = self.activation(x)
        x = self.pool1(x)
        x = self.dropout1(x)
        x = self.separable(x)
        x = self.batchnorm3(x)
        x = self.activation(x)
        x = self.pool2(x)
        x = self.dropout2(x)
        return x

    def forward(self, x):
        x = self._features(x)
        x = x.view(x.size(0), -1)
        x = self.classify(x)
        return x

## 4. Train models and compare scenarios

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y1)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)

X1_tr_t = torch.tensor(X1_tr, dtype=torch.float32).unsqueeze(1)
X1_te_t = torch.tensor(X1_te, dtype=torch.float32).unsqueeze(1)
X2_tr_t = torch.tensor(X2_tr, dtype=torch.float32).unsqueeze(1)
X2_te_t = torch.tensor(X2_te, dtype=torch.float32).unsqueeze(1)
y1_tr_t = torch.tensor(y1_tr, dtype=torch.long)
y2_tr_t = torch.tensor(y2_tr, dtype=torch.long)

n_ch, n_s = X1.shape[1], X1.shape[2]

def train_model(model, X_tr, y_tr, epochs=30, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        perm = torch.randperm(X_tr.shape[0])
        for s in range(0, X_tr.shape[0], 32):
            idx = perm[s:s+32]
            opt.zero_grad()
            loss = criterion(model(X_tr[idx].to(device)), y_tr[idx].to(device))
            loss.backward()
            opt.step()

def eval_model(model, X_te, y_te):
    model.eval()
    with torch.no_grad():
        _, pred = torch.max(model(X_te.to(device)), 1)
    return accuracy_score(y_te, pred.cpu().numpy())

print("Training on S1...")
model_s1 = EEGNet(n_channels=n_ch, n_samples=n_s).to(device)
train_model(model_s1, X1_tr_t, y1_tr_t, epochs=30)
acc_zeroshot = eval_model(model_s1, X2_te_t, y2_te)
print(f"Zero-shot S1->S2: {acc_zeroshot:.4f}")

print("Fine-tuning on S2...")
model_ft = EEGNet(n_channels=n_ch, n_samples=n_s).to(device)
model_ft.load_state_dict(model_s1.state_dict())
train_model(model_ft, X2_tr_t, y2_tr_t, epochs=10, lr=0.0001)
acc_finetune = eval_model(model_ft, X2_te_t, y2_te)
print(f"Fine-tuned: {acc_finetune:.4f}")

torch.manual_seed(42)
np.random.seed(42)
print("Training S2 from scratch...")
model_scratch = EEGNet(n_channels=n_ch, n_samples=n_s).to(device)
train_model(model_scratch, X2_tr_t, y2_tr_t, epochs=30)
acc_scratch = eval_model(model_scratch, X2_te_t, y2_te)
print(f"From scratch: {acc_scratch:.4f}")

## 5. Interactive plot

In [ ]:
import plotly.graph_objects as go

scenarios = ['Zero-shot<br>S1->S2', 'Fine-tune<br>S1->S2', 'From scratch<br>S2']
accs = [acc_zeroshot, acc_finetune, acc_scratch]
colors = ['steelblue', 'coral', 'seagreen']

fig = go.Figure(go.Bar(x=scenarios, y=accs, marker_color=colors, text=[f'{a:.4f}' for a in accs], textposition='outside'))
fig.update_layout(title='Transfer Learning: EEGNet Between Subjects', yaxis_title='Accuracy on Subject 2', yaxis_range=[0, 1], width=700, height=500)
fig.show()

## What did we learn?
- Zero-shot transfer is weak due to inter-subject variability
- Fine-tuning improves performance moderately
- Training from scratch is best with sufficient data